# Zero-Train Optimization & Maintenance (ZTOM) of a Legal Understanding LLM

**Legal Understanding LLM — [Google Gemma 3 1B IT](https://huggingface.co/google/gemma-3-1b-it) + LoRA (PEFT)**

## Model setup

- **Base model:** [Gemma 3 1B IT](https://huggingface.co/google/gemma-3-1b-it) — (instruction-tuned)
- **Adaptation:** LoRA adapters via [Hugging Face PEFT](https://huggingface.co/docs/peft)
- **Rationale:** Efficient training, standard practice, and straightforward comparison across checkpoints

## Fine-tuning tasks

### Task A — Legal reasoning

Use datasets that already pair inputs with **Yes/No** (or binary) answers where applicable:
- **[nguha/legalbench](https://huggingface.co/datasets/nguha/legalbench)** — LegalBench: many tasks over legal text; see the [dataset card](https://huggingface.co/datasets/nguha/legalbench) and [project site](https://hazyresearch.stanford.edu/legalbench/).
- **[chenghao/cuad_qa](https://huggingface.co/datasets/chenghao/cuad_qa)** — CUAD as QA over contracts; [CUAD paper](https://arxiv.org/abs/2103.06268).

### Task B — Summarization

- **Dataset:** [BillSum](https://huggingface.co/datasets/billsum) — long bills with human summaries.
- **I/O:** long bill or document text → concise summary.
- **Evaluation:** **ROUGE** (e.g. ROUGE-1 / ROUGE-L) vs. reference summaries.

## Scenario

After initial training, the model may benefit from optimization to improve its response quality and accuracy. Traditional approaches would require retraining the model with additional data, which is time-consuming and computationally expensive. ZTOM provides an alternative solution that optimizes the model's performance using only a small validation set and semantic similarity metrics to guide the optimization process.

## Use Case

This demonstration is applicable when:
- A model is already trained and deployed, but performance improvements are desired
- Retraining is not feasible due to time, computational, or data constraints
- You have access to only a small validation set for optimization
- You want to improve response quality using semantic similarity as a guide

## Summary

This notebook shows how **Zero-Train Optimization & Maintenance (ZTOM)** adjusts the Legal Understanding LLM (Gemma 3 1B IT + LoRA) **without retraining**. The objective is **mean string similarity** between model outputs and reference answers (RapidFuzz `partial_ratio`, higher is better), with **`minimize=False`** so ZTOM **maximizes** that score. In this run, the score rose from **~31.2** to **~59.7** (**about +91% vs. the starting value**, or **~1.9×** the baseline), using **100** optimizer evaluations on the validation slice used in the notebook.
  
## ZTOM Result Outputs

| Metric | Value |
|--------|------:|
| **Objective** | Mean RapidFuzz `partial_ratio` vs. `dataset["answer"]` (higher is better) |
| **Optimizer** | `minimize=False` (maximize objective) |
| **Scaling factors** | `[-0.262, 0.084, 0.999, -0.614, -0.226, 0.765, -0.215, 0.577, 0.276]` |
| **Original objective** | 31.227872848510742 |
| **Best objective** | 59.72260665893555 |
| **Relative improvement** | ~91% vs. original \((\text{best}-\text{original})/\text{original}\) |
| **Number of evaluations** | 100 |

### Install dependencies and utility functions: This cell should be run once.

In [1]:
%%capture
%pip install -r requirements.txt
%pip install 'authentrics==0.21.2' --extra-index-url='https://us-central1-python.pkg.dev/authentrics/authentrics/simple'

In [2]:
from authentrics import AuthentricsSession
from dotenv import load_dotenv
from tokenizers import Tokenizer
import pathlib as Path


load_dotenv()

def cleanup_project(session: AuthentricsSession, project_name: str):
    print("Cleaning up project: ", project_name)
    if Path(project_name).exists():
        try:
            projects = session.get_projects()
            print("Projects: found")
            for project in projects:
                if project.name == project_name:
                    session.delete_project(project)
                    print("Project: deleted")
                    Path(project_name).rmdir()
                    print("Project: removed")
                    return
        except Exception as e:
            print("Error: ", e)
            Path(project_name).rmdir()


def format_row(tokenizer: Tokenizer, example):
    messages = [
        {
            "role": "user",
            "content": (
                "Answer the legal question based on the contract.\n\n"
                f"Question: {example['question']}\n\n"
                f"Context:\n{example['context']}"
            ),
        },
        {"role": "assistant", "content": example["answer"]},
    ]
    example["formatted_chat"] = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return example

## Download Checkpoints and Dataset

In [3]:
from pathlib import Path

checkpoint_dir = Path('checkpoints')
data_dir = Path('data')

checkpoint_dir.mkdir(parents=True, exist_ok=True)
data_dir.mkdir(parents=True, exist_ok=True)

# !unlink checkpoints/cuad_checkpoints
# !unlink checkpoints/bill_sum_v2
# !unlink data/test_cuad_qa.json
# !ln -s /tmp/draft-webinar-demo/checkpoints/cuad_checkpoints checkpoints/cuad_checkpoints
# !ln -s /tmp/draft-webinar-demo/checkpoints/bill_sum_v2 checkpoints/bill_sum_v2
# !ln -s /tmp/draft-webinar-demo/datasets/test_cuad_qa.json data/test_cuad_qa.json

checkpoints = [checkpoint_dir / "cuad_checkpoints" / f"checkpoint-{i}" for i in range(300, 1201, 300)]
checkpoints.append(checkpoint_dir / "cuad_checkpoints" / "final_model" / "checkpoint_20260311_231438")
checkpoints.append(checkpoint_dir / "bill-sum-train-220260330-1720" / "checkpoint-300")
checkpoints.append(checkpoint_dir / "bill-sum-train-220260330-1720" / "checkpoint-600")
# checkpoints.append(checkpoint_dir / "bill-sum-train-220260330-1720" / "checkpoint-425")
checkpoints.append(checkpoint_dir / "bill-sum-train-220260330-1720" / "final_model" / "checkpoint_20260330_225944")
# checkpoints.append(checkpoint_dir / "bill_sum_20260324-215955" / "checkpoint-200")
# checkpoints.append(checkpoint_dir / "bill_sum_20260324-215955" / "checkpoint-400")
# checkpoints.append(checkpoint_dir / "bill_sum_20260324-215955" / "checkpoint-471")
# checkpoints.append(checkpoint_dir / "bill_sum_20260324-215955" / "final_model" / "checkpoint_20260325_004149")
# checkpoints += [checkpoint_dir / "bill_sum" / f"checkpoint-{i}" for i in range(300, 1201, 300)]
# checkpoints.append(checkpoint_dir / "bill_sum" / "final_model" / "checkpoint_20260315_180509")
# checkpoints += [checkpoint_dir / "bill_sum_v2" / f"checkpoint-{i}" for i in range(300, 1201, 300)]
# checkpoints.append(checkpoint_dir / "bill_sum_v2" / "final_model" / "checkpoint_20260315_180509")

cuad_data_file = data_dir / "test_cuad_qa.json"
print(cuad_data_file)
bill_sum_data_file = data_dir / "bill_summary_test.json"
print(bill_sum_data_file)
print(checkpoints)

data/test_cuad_qa.json
data/bill_summary_test.json
[PosixPath('checkpoints/cuad_checkpoints/checkpoint-300'), PosixPath('checkpoints/cuad_checkpoints/checkpoint-600'), PosixPath('checkpoints/cuad_checkpoints/checkpoint-900'), PosixPath('checkpoints/cuad_checkpoints/checkpoint-1200'), PosixPath('checkpoints/cuad_checkpoints/final_model/checkpoint_20260311_231438'), PosixPath('checkpoints/bill-sum-train-220260330-1720/checkpoint-300'), PosixPath('checkpoints/bill-sum-train-220260330-1720/checkpoint-600'), PosixPath('checkpoints/bill-sum-train-220260330-1720/final_model/checkpoint_20260330_225944')]


## Authentrics Python Library

In [4]:
from authentrics import AuthentricsException, AuthentricsSession, ZtomOptimizationOptions
from datetime import datetime

PROJECT_NAME = "ZTOM_Local_"
PROJECT_NAME = PROJECT_NAME + str(int(datetime.now().timestamp()))
project_path = Path(PROJECT_NAME)

session = AuthentricsSession()
session.login()

try:
    project = session.load_project(project_path)

except AuthentricsException:
    cleanup_project(session, PROJECT_NAME)
    project = session.create_project(project_path, PROJECT_NAME)
    print(checkpoints[-1])
    project = session.add_checkpoints(project, *checkpoints)


Cleaning up project:  ZTOM_Local_1774979424
[DEBUG] Status code: 200
checkpoints/bill-sum-train-220260330-1720/final_model/checkpoint_20260330_225944
[INFO] Logged in successfully
[DEBUG] Making POST request to /project
[DEBUG] Body: {"description":"","format":"ONNX","name":"ZTOM_Local_1774979424","type":"CLIENT_MANAGED"}
[DEBUG] Status code: 200
[DEBUG] Creating checkpoint for project 69cc096b690c8e290bed8854 with filename checkpoint-300 and hash 58271e6bd4d0f6e1a1bd906b6bff5b0f1800e6766e15156026601f623b24fcc1
[DEBUG] Making POST request to /project/file/external
[DEBUG] Body: {"fileName":"checkpoint-300","filePath":"LOCAL","format":"ONNX","hash":"58271e6bd4d0f6e1a1bd906b6bff5b0f1800e6766e15156026601f623b24fcc1","projectId":"69cc096b690c8e290bed8854"}
[DEBUG] Status code: 200
[DEBUG] Successfully created checkpoint: {"id":"69cc096b690c8e290bed8854","name":"ZTOM_Local_1774979424","description":"","format":"ONNX","createdOn":1774979435620,"modifiedOn":null,"baseModel":null,"type":"CLIEN

In [5]:
project

Project(id=69cc096b690c8e290bed8854, name='ZTOM_Local_1774979424', description='', created_at='Tue Mar 31 17:50:35.620000000 2026', project_path='optional("/home/seanhagstrom/core/demos/ZTOM_Local/ZTOM_Local_1774979424")')

In [6]:
project.checkpoints

[Checkpoint(id=69cc096e690c8e290bed8856, name='checkpoint-300', hash=58271e6bd4d0f6e1a1bd906b6bff5b0f1800e6766e15156026601f623b24fcc1, created_at='Tue Mar 31 17:50:38.406000000 2026', path='optional("/tmp/draft-webinar-demo/checkpoints/cuad_checkpoints/checkpoint-300")'),
 Checkpoint(id=69cc0971690c8e290bed8857, name='checkpoint-600', hash=6555d5a9df97f95368fbee731a797a3f74e1e81288eba44718e59a69af4359e6, created_at='Tue Mar 31 17:50:41.176000000 2026', path='optional("/tmp/draft-webinar-demo/checkpoints/cuad_checkpoints/checkpoint-600")'),
 Checkpoint(id=69cc0974690c8e290bed8858, name='checkpoint-900', hash=ba95fa6a4b5892f36f94b554cd2a7f3b110f1b8ae4c7c21903ba2929aafa6ada, created_at='Tue Mar 31 17:50:44.021000000 2026', path='optional("/tmp/draft-webinar-demo/checkpoints/cuad_checkpoints/checkpoint-900")'),
 Checkpoint(id=69cc0976690c8e290bed8859, name='checkpoint-1200', hash=3a1a5d5dd8415d656acfc18675a850f50def3185135245c6ed46d9b07c8ad1b7, created_at='Tue Mar 31 17:50:46.783000000 202

### Model Wrapper

In [7]:
from authentrics import InferenceResult, ModelInterface, Parameters, WeightBias, use_backend
from datasets import Column, Dataset
from transformers import TextGenerationPipeline
from transformers.pipelines import pipeline


class SimpleHFModel(ModelInterface):
    def __init__(self, dataset: Dataset | None = None, batch_size: int = 1):
        super().__init__()

        use_backend("torch")

        self._dataset = dataset or Dataset.from_list([])
        self._batch_size = batch_size

        self._inference_config = {"max_new_tokens": 1024, "do_sample": False, "return_full_text": False}

        self._module = None
        self._input_data = None

    def load(self, checkpoint_path: Path | str | bytes) -> None:
        self._module: TextGenerationPipeline = pipeline(
            "text-generation",
            model=str(checkpoint_path),
            trust_remote_code=True,
            # device_map="sequential",
        )

        if self._input_data is None:
            formatted_chat: Column = self._dataset.map(
                lambda example: format_row(self._module.tokenizer, example)
            )["formatted_chat"]
            self._input_data = list(formatted_chat)

    def get_weight_bias(
        self,
        weight_names: list[str] | None = None,
        bias_names: list[str] | None = None,
    ) -> WeightBias:
        weights = Parameters()
        biases = Parameters()
        for name, param in self._module.model.named_parameters():
            last_part = name.rsplit(".", 1)[-1]
            if last_part == "weight":
                if weight_names is None or name in weight_names:
                    weights[name] = param.detach().cpu()

            elif last_part == "bias":
                if bias_names is None or name in bias_names:
                    biases[name] = param.detach().cpu()

        return WeightBias(weights, biases)

    def perform_inference(
        self,
        return_intermediate_outputs: bool = False,
        layer_names: list[str] | None = None,
    ) -> InferenceResult:
        chat_template = self._inference_config.pop("chat_template", None)

        assert self._module.tokenizer is not None
        if chat_template is not None:
            self._module.tokenizer.chat_template = chat_template

        result = self._module(
            text_inputs=self._input_data,
            batch_size=self._batch_size,
            chat_template=chat_template,
            **self._inference_config,
        )

        return InferenceResult([r[0]["generated_text"] for r in result])

    def set_weight_bias(self, weight_bias: WeightBias) -> None:
        for name, tensor in self._module.model.named_parameters():
            if name in weight_bias.weights:
                tensor.data.copy_(weight_bias.weights[name])
            if name in weight_bias.biases:
                tensor.data.copy_(weight_bias.biases[name])

    def save(self, checkpoint_path: Path | str | bytes) -> None:
        path = Path(checkpoint_path)
        path.parent.mkdir(parents=True, exist_ok=True)

        # Avoid a bug in the Hugging Face pipeline
        if not hasattr(self._module, "modelcard"):
            self._module.modelcard = None

        self._module.save_pretrained(path)


In [8]:
from __future__ import annotations

from typing import Optional, Any

import torch
from peft import PeftModel
from rapidfuzz import fuzz
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline


def clean_answer(text):
    if "<start_of_turn>model" in text:
        text = text.split("<start_of_turn>model")[-1]

    return text.strip()


def _tokenize(text: str) -> list[str]:
    return text.lower().split()


def _lcs_length(a: list[str], b: list[str]) -> int:
    if not a or not b:
        return 0

    prev = [0] * (len(b) + 1)
    for token_a in a:
        curr = [0] * (len(b) + 1)
        for j, token_b in enumerate(b, start=1):
            if token_a == token_b:
                curr[j] = prev[j - 1] + 1
            else:
                curr[j] = max(prev[j], curr[j - 1])
        prev = curr
    return prev[-1]


def _rouge_l_scores(prediction: str, reference: str) -> dict[str, float]:
    pred_tokens = _tokenize(prediction)
    ref_tokens = _tokenize(reference)
    if not pred_tokens or not ref_tokens:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}

    lcs = _lcs_length(pred_tokens, ref_tokens)
    precision = lcs / len(pred_tokens)
    recall = lcs / len(ref_tokens)
    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = (2 * precision * recall) / (precision + recall)

    return {"precision": precision, "recall": recall, "f1": f1}


def evaluate_qa_model(pipe, dataset, num_samples: Optional[int] = None, max_new_tokens: int = 1024):
    correct = 0
    total = 0
    partial_ratio_sum = 0.0
    exact_match_count = 0

    for i, row in enumerate(dataset):
        if num_samples and i >= num_samples:
            break

        prompt = row["formatted_chat"]
        gold = row["answer"]

        result = pipe(prompt, max_new_tokens=max_new_tokens, do_sample=False, return_full_text=False)
        generated = result[0]["generated_text"]

        pred = clean_answer(generated)
        score = fuzz.partial_ratio(pred.lower(), gold.lower())

        partial_ratio_sum += score
        if pred.lower() == gold.lower():
            exact_match_count += 1
        if score >= 90:
            correct += 1
        total += 1

    if total == 0:
        return {
            "analysis_accuracy": 0.0,
            "analysis_exact_match": 0.0,
            "analysis_avg_partial_ratio": 0.0,
            "analysis_samples": 0.0,
        }

    return {
        "analysis_accuracy": correct / total,
        "analysis_exact_match": exact_match_count / total,
        "analysis_avg_partial_ratio": partial_ratio_sum / total,
        "analysis_samples": float(total),
    }


def evaluate_summarization_model(
    pipe,
    dataset,
    num_samples: Optional[int] = None,
    max_new_tokens: int = 1024,
):
    total = 0
    rouge_l_precision_sum = 0.0
    rouge_l_recall_sum = 0.0
    rouge_l_f1_sum = 0.0
    partial_ratio_sum = 0.0

    for i, row in enumerate(dataset):
        if num_samples and i >= num_samples:
            break

        prompt = row["formatted_chat"]
        gold = row["summary"]

        result = pipe(prompt, max_new_tokens=max_new_tokens, do_sample=False, return_full_text=False)
        generated = result[0]["generated_text"]

        pred = clean_answer(generated)
        rouge_scores = _rouge_l_scores(pred, gold)

        rouge_l_precision_sum += rouge_scores["precision"]
        rouge_l_recall_sum += rouge_scores["recall"]
        rouge_l_f1_sum += rouge_scores["f1"]
        partial_ratio_sum += fuzz.partial_ratio(pred.lower(), gold.lower())
        total += 1

    if total == 0:
        return {
            "analysis_rouge_l_precision": 0.0,
            "analysis_rouge_l_recall": 0.0,
            "analysis_rouge_l_f1": 0.0,
            "analysis_avg_partial_ratio": 0.0,
            "analysis_samples": 0.0,
        }

    return {
        "analysis_rouge_l_precision": rouge_l_precision_sum / total,
        "analysis_rouge_l_recall": rouge_l_recall_sum / total,
        "analysis_rouge_l_f1": rouge_l_f1_sum / total,
        "analysis_avg_partial_ratio": partial_ratio_sum / total,
        "analysis_samples": float(total),
    }


def prefix_metrics(metrics: dict[str, float], prefix: str) -> dict[str, float]:
    return {f"{prefix}{key}": value for key, value in metrics.items()}

def _torch_dtype() -> torch.dtype:
    if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
        return torch.bfloat16
    return torch.float16

def _build_model_kwargs(enable_4bit: bool) -> dict:
    dtype = _torch_dtype()
    model_kwargs = dict(attn_implementation="eager", torch_dtype=dtype)

    if enable_4bit:
        model_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=dtype,
            bnb_4bit_quant_storage=dtype,
        )

    return model_kwargs

def load_finetuned_model(model_id: str, model_path: str, enable_4bit: bool):
    model_kwargs = _build_model_kwargs(enable_4bit)
    adapter_config_path = Path(model_path) / "adapter_config.json"
    base_model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)

    if adapter_config_path.exists():
        return PeftModel.from_pretrained(base_model, model_path)

    return AutoModelForCausalLM.from_pretrained(model_path, **model_kwargs)

def build_text_generation_pipeline(model, tokenizer):
    return pipeline("text-generation", model=model, tokenizer=tokenizer)

def evaluate_model(
*,
task: str,
tokenizer: Any,
formatted_dataset: Any,
model,
metrics_prefix: str = "",
num_samples: Optional[int] = None,
max_new_tokens: int = 1024,
) -> dict[str, float]:
    text_generation = build_text_generation_pipeline(model, tokenizer)

    if task == "cuad":
        metrics = evaluate_qa_model(
            text_generation,
            formatted_dataset,
            num_samples=num_samples,
            max_new_tokens=max_new_tokens,
        )
        return prefix_metrics(metrics, metrics_prefix)

    if task == "bill_sum":
        metrics = evaluate_summarization_model(
            text_generation,
            formatted_dataset,
            num_samples=num_samples,
            max_new_tokens=max_new_tokens,
        )
        return prefix_metrics(metrics, metrics_prefix)

    raise RuntimeError(f"Unknown analysis task: {task}")

def evaluate_finetuned_model(
    *,
    task: str,
    model_id: str,
    model_path: str,
    tokenizer: Any,
    formatted_dataset: Any,
    enable_4bit: bool,
    metrics_prefix: str = "finetuned_",
    num_samples: Optional[int] = None,
    max_new_tokens: int = 1024,
) -> dict[str, float]:
    model = load_finetuned_model(model_id, model_path, enable_4bit=enable_4bit)
    return evaluate_model(
        task=task,
        tokenizer=tokenizer,
        formatted_dataset=formatted_dataset,
        model=model,
        metrics_prefix=metrics_prefix,
        num_samples=num_samples,
        max_new_tokens=max_new_tokens,
    )

In [9]:

def _format_cuad_dataset(dataset, tokenizer, use_base_prompt: bool):
    def format_row(example):
        if use_base_prompt:
            instruction = (
                "Answer the legal question based on the contract.\n"
                "Respond with only the answer phrase."
            )
        else:
            instruction = "Answer the legal question based on the contract."

        example["formatted_chat"] = tokenizer.apply_chat_template(
            [
                {
                    "role": "user",
                    "content": (
                        f"{instruction}\n\n"
                        f"Question: {example['question']}\n\n"
                        f"Context:\n{example['context']}"
                    ),
                }
            ],
            tokenize=False,
            add_generation_prompt=True,
        )
        return example

    return dataset.map(format_row)


def _format_bill_sum_dataset(dataset, tokenizer):
    def format_row(example):
        return {
            "formatted_chat": tokenizer.apply_chat_template(
                [
                    {
                        "role": "user",
                        "content": f"Summarize the document.\n\nContext:\n{example['text']}",
                    }
                ],
                tokenize=False,
                add_generation_prompt=True,
            ),
            "summary": example["summary"],
        }

    return dataset.map(format_row, remove_columns=dataset.column_names)

### Prepare the Data

In [10]:
from datasets import load_dataset


dataset = load_dataset(
    "json",
    data_files={"eval": str(cuad_data_file)},
)["eval"].take(100)

bill_sum_dataset = load_dataset(
    "json",
    data_files={"eval": str(bill_sum_data_file)},
)["eval"].take(100)
print(bill_sum_dataset)




Dataset({
    features: ['text', 'summary'],
    num_rows: 100
})


Compare cuad checkpoints and bill sum checkpoints

In [11]:
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-it")
tokenizer.pad_token = tokenizer.eos_token

cuad_dataset = _format_cuad_dataset(dataset, tokenizer, use_base_prompt=False)
bill_sum_dataset = _format_bill_sum_dataset(bill_sum_dataset, tokenizer)

In [12]:
%%time

# tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-it")
# tokenizer.pad_token = tokenizer.eos_token

# cuad_dataset = _format_cuad_dataset(dataset, tokenizer, use_base_prompt=False)
# bill_sum_dataset = _format_bill_sum_dataset(bill_sum_dataset, tokenizer)

cuad_result = evaluate_finetuned_model(
    task="cuad",
    model_id="google/gemma-3-1b-it",
    model_path="checkpoints/cuad_checkpoints/final_model/checkpoint_20260311_231438",
    tokenizer=tokenizer,
    formatted_dataset=cuad_dataset,
    enable_4bit=True,
    num_samples=100,
    max_new_tokens=1024
)

print("cuad_result", cuad_result)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_to

cuad_result {'finetuned_analysis_accuracy': 0.92, 'finetuned_analysis_exact_match': 0.76, 'finetuned_analysis_avg_partial_ratio': 95.97124484456828, 'finetuned_analysis_samples': 100.0}
CPU times: user 12min 20s, sys: 952 ms, total: 12min 21s
Wall time: 12min 18s


In [13]:
bill_sum_cuad_result = evaluate_finetuned_model(
    task="cuad",
    model_id="google/gemma-3-1b-it",
    model_path="checkpoints/bill-sum-train-220260330-1720/final_model/checkpoint_20260330_225944",
    tokenizer=tokenizer,
    formatted_dataset=cuad_dataset,
    enable_4bit=True,
    num_samples=100,
    max_new_tokens=1024
)

print("bill_sum_cuad_result", bill_sum_cuad_result)

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


bill_sum_cuad_result {'finetuned_analysis_accuracy': 0.25, 'finetuned_analysis_exact_match': 0.05, 'finetuned_analysis_avg_partial_ratio': 65.69469820050905, 'finetuned_analysis_samples': 100.0}


In [14]:
bill_sum_result = evaluate_finetuned_model(
    task="bill_sum",
    model_id="google/gemma-3-1b-it",
    model_path="checkpoints/bill-sum-train-220260330-1720/final_model/checkpoint_20260330_225944",
    tokenizer=tokenizer,
    formatted_dataset=bill_sum_dataset,
    enable_4bit=True,
    num_samples=100,
    max_new_tokens=1024
)
print("bill_sum_result", bill_sum_result)

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


bill_sum_result {'finetuned_analysis_rouge_l_precision': 0.6190755862958255, 'finetuned_analysis_rouge_l_recall': 0.5346203610508925, 'finetuned_analysis_rouge_l_f1': 0.5210313288643671, 'finetuned_analysis_avg_partial_ratio': 76.23044755545412, 'finetuned_analysis_samples': 100.0}


In [15]:
print("cuad_result", cuad_result)
print("bill_sum_cuad_result", bill_sum_cuad_result)
print("bill_sum_result", bill_sum_result)


cuad_result {'finetuned_analysis_accuracy': 0.92, 'finetuned_analysis_exact_match': 0.76, 'finetuned_analysis_avg_partial_ratio': 95.97124484456828, 'finetuned_analysis_samples': 100.0}
bill_sum_cuad_result {'finetuned_analysis_accuracy': 0.25, 'finetuned_analysis_exact_match': 0.05, 'finetuned_analysis_avg_partial_ratio': 65.69469820050905, 'finetuned_analysis_samples': 100.0}
bill_sum_result {'finetuned_analysis_rouge_l_precision': 0.6190755862958255, 'finetuned_analysis_rouge_l_recall': 0.5346203610508925, 'finetuned_analysis_rouge_l_f1': 0.5210313288643671, 'finetuned_analysis_avg_partial_ratio': 76.23044755545412, 'finetuned_analysis_samples': 100.0}


### Run ZTOM Model Maintenance

#### Setup loss function

This is a custom loss function that calculates the average fuzzy match between the predicted and actual answers. We will maximize this function over the course of the optimization process.

In [16]:
import torch
from rapidfuzz import fuzz


def clean_answer(text: str) -> str:
    if "<start_of_turn>model" in text:
        text = text.split("<start_of_turn>model")[-1]

    return text.strip()


def average_fuzzy_match(y_pred: list[str], answers: list[str]) -> float:
    if len(y_pred) != len(answers):
        raise ValueError("y_pred and answers must have the same length")

    total = len(y_pred)
    if total == 0:
        return 0.0

    scores = torch.tensor(
        [
            fuzz.partial_ratio(clean_answer(output).lower(), answer.lower())
            for output, answer in zip(y_pred, answers)
        ]
    )

    return scores.mean().item()


In [17]:
%%time
from transformers import logging
session.login()

optimized_checkpoint_name = "optimized_checkpoint_" + str(int(datetime.now().timestamp()))

logging.set_verbosity(logging.ERROR)

session.model = SimpleHFModel(dataset=dataset, batch_size=10)

options = ZtomOptimizationOptions(
    max_evaluations=100,
    xtol_rel=1e-4,
    ftol_rel=1e-4,
    lower_bound=-1.0,
    upper_bound=1.0,
    minimize=False,
)

ztom_result = session.ztom_analysis(
    project,
    lambda y_pred: average_fuzzy_match(y_pred, dataset["answer"]),
    project_path / optimized_checkpoint_name,
    options,
)

print(ztom_result)

[DEBUG] Status code: 200
[INFO] Logged in successfully
[INFO] Registered tensor factory for backend: torch
[DEBUG] Making GET request to /api/auth/user
[DEBUG] Status code: 200
[DEBUG] Making POST request to /api/auth/user/permission
[DEBUG] Body: {"endpoint":"/zto/batch","projectId":"69cc096b690c8e290bed8854"}
[DEBUG] Status code: 200
[INFO] Permission Granted for user:, endpoint:/zto/batch, grantedBy: ProjectAuthorizationManager, grantedAt: 1774982833261, projectId:69cc096b690c8e290bed8854
[DEBUG] Making GET request to /project/69cc096b690c8e290bed8854
[DEBUG] Status code: 200


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/364 [00:00<?, ?it/s]

Parameter 'function'=<function SimpleHFModel.load.<locals>.<lambda> at 0x793993b7aa20> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only shown once. Subsequent hashing failures won't be shown.


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

...

Loading weights:   0%|          | 0/364 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/364 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Optimized checkpoint path: ZTOM_Local_1774979424/optimized_checkpoint_1774982827, Scaling factors: [0.5470733811800851, -0.13674356159827672, 0.5561636971703701, 0.09921116817512865, 0.09514423754513346, 0.03799202361065904, -0.04887588245324496], Original loss: 44.91648483276367, Best loss: 61.2156982421875, Number of inferences: 97
CPU times: user 13h 32min 24s, sys: 10min 54s, total: 13h 43min 18s
Wall time: 13h 7min 4s


In [18]:
optimized_checkpoint_path = project_path / optimized_checkpoint_name
optimized_cuad_result = evaluate_finetuned_model(
    task="cuad",
    model_id="google/gemma-3-1b-it",
    model_path=optimized_checkpoint_path,
    tokenizer=tokenizer,
    formatted_dataset=cuad_dataset,
    enable_4bit=True,
    num_samples=100,
    max_new_tokens=1024
)

print("optimized_cuad_result", optimized_cuad_result)

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

optimized_cuad_result {'finetuned_analysis_accuracy': 0.16, 'finetuned_analysis_exact_match': 0.0, 'finetuned_analysis_avg_partial_ratio': 58.01364958528482, 'finetuned_analysis_samples': 100.0}


In [19]:
optimized_bill_sum_result = evaluate_finetuned_model(
    task="bill_sum",
    model_id="google/gemma-3-1b-it",
    model_path=optimized_checkpoint_path,
    tokenizer=tokenizer,
    formatted_dataset=bill_sum_dataset,
    enable_4bit=True,
    num_samples=100,
    max_new_tokens=1024
)
print("optimized_bill_sum_result", optimized_bill_sum_result)

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

optimized_bill_sum_result {'finetuned_analysis_rouge_l_precision': 0.5309677193498762, 'finetuned_analysis_rouge_l_recall': 0.5625545885988966, 'finetuned_analysis_rouge_l_f1': 0.5033469805674481, 'finetuned_analysis_avg_partial_ratio': 74.07886437579533, 'finetuned_analysis_samples': 100.0}


In [20]:
print("optimized_cuad_result", optimized_cuad_result)
print("optimized_bill_sum_result", optimized_bill_sum_result)

optimized_cuad_result {'finetuned_analysis_accuracy': 0.16, 'finetuned_analysis_exact_match': 0.0, 'finetuned_analysis_avg_partial_ratio': 58.01364958528482, 'finetuned_analysis_samples': 100.0}
optimized_bill_sum_result {'finetuned_analysis_rouge_l_precision': 0.5309677193498762, 'finetuned_analysis_rouge_l_recall': 0.5625545885988966, 'finetuned_analysis_rouge_l_f1': 0.5033469805674481, 'finetuned_analysis_avg_partial_ratio': 74.07886437579533, 'finetuned_analysis_samples': 100.0}


In [21]:
# session.login()

# project = session.load_project("ZTOM_Local_Sean")
# prev = project.checkpoints[-6]
# curr = project.checkpoints[-1]
# print(prev.path)
# print(curr.path)
# static_analysis_result = session.static_analysis(
#     project,
#     prev,
#     curr,
# )

# print(static_analysis_result)

In [22]:
# session.login()
# session.model = SimpleHFModel(dataset=dataset, batch_size=10)

# project = session.load_project("ZTOM_Local_Sean_2")
# prev = project.checkpoints[-3]
# curr = project.checkpoints[-1]
# print(prev.path)
# print(curr.path)
# static_analysis_result = session.static_analysis(
#     project,
#     prev,
#     curr,
# )

# print(static_analysis_result)

In [23]:
# session.login()
# session.model = SimpleHFModel(dataset=dataset, batch_size=10)

# project = session.load_project("ZTOM_Local_Sean_3")
# prev = project.checkpoints[-5]
# curr = project.checkpoints[-4]
# print(prev.path)
# print(curr.path)
# static_analysis_result = session.static_analysis(
#     project,
#     prev,
#     curr,
# )

# print(static_analysis_result)

In [24]:
# session.login()
# session.model = SimpleHFModel(dataset=dataset, batch_size=10)

# project = session.load_project("ZTOM_Local_Sean_0331")
# prev = project.checkpoints[-2]
# curr = project.checkpoints[-1]
# print(prev.path)
# print(curr.path)
# static_analysis_result = session.static_analysis(
#     project,
#     prev,
#     curr,
# )

# print(static_analysis_result)